# Decision Tree Assignment — Information Gain & Feature Selection

This notebook shows **exactly how the decision tree is built** from the 20 game records.

It will show:
- which Record IDs are at the current node
- every possible feature split
- the branches created by each feature
- Record IDs in each branch
- Hit/Flop counts
- entropy of each branch
- branch weight
- weighted entropy contribution
- Information Gain for every feature
- which feature has the highest Information Gain
- the final decision tree
- the path each Record ID takes through the final tree


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree


## 2. Load the CSV file

In [ ]:
df = pd.read_csv("data/game_success.csv")

df


## 3. Check the columns and target counts

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nNumber of records:", len(df))

print("\nHit/Flop counts:")
print(df["Hit/Flop"].value_counts())


## 4. Entropy function

Entropy measures how mixed the classes are at a node.

- Entropy = 0 means the node is pure.
- Higher entropy means the node contains a greater mixture of Hit and Flop records.


In [ ]:
def entropy(data, target="Hit/Flop"):
    counts = data[target].value_counts()
    total = len(data)

    entropy_value = 0

    for count in counts:
        probability = count / total
        entropy_value -= probability * np.log2(probability)

    return entropy_value


## 5. Starting entropy of all 20 records

In [ ]:
print("Record IDs:", df["Record ID"].tolist())
print("\nClass counts:")
print(df["Hit/Flop"].value_counts())
print(f"\nStarting Entropy = {entropy(df):.4f}")


## 6. Function to compare every possible split

This is the main function for the assignment. It prints every branch created by every candidate feature and then identifies the feature with the highest Information Gain.

In [ ]:
def compare_splits(data, features, target="Hit/Flop"):

    print("=" * 100)
    print("CURRENT NODE")
    print("=" * 100)

    print("\nRecord IDs entering this node:")
    print(data["Record ID"].tolist())

    print("\nClass counts:")
    print(data[target].value_counts())

    parent_entropy = entropy(data, target)
    print(f"\nTotal Records = {len(data)}")
    print(f"Parent Entropy = {parent_entropy:.4f}")

    results = []

    for feature in features:
        print("\n\n" + "#" * 100)
        print(f"TESTING FEATURE: {feature}")
        print("#" * 100)

        weighted_entropy = 0

        for value in data[feature].unique():
            subset = data[data[feature] == value]

            branch_entropy = entropy(subset, target)
            weight = len(subset) / len(data)
            weighted_value = weight * branch_entropy
            weighted_entropy += weighted_value

            print("\n" + "-" * 90)
            print(f"BRANCH: {feature} = {value}")
            print("-" * 90)

            print("\nRecord IDs in this branch:")
            print(subset["Record ID"].tolist())

            print("\nRecords in this branch:")
            print(
                subset[["Record ID", feature, target]].to_string(index=False)
            )

            print("\nClass Counts:")
            print(subset[target].value_counts())

            print(f"\nNumber of Records = {len(subset)}")
            print(f"Weight = {len(subset)}/{len(data)} = {weight:.4f}")
            print(f"Entropy = {branch_entropy:.4f}")
            print(
                f"Weighted Entropy Contribution = "
                f"{weight:.4f} x {branch_entropy:.4f} = {weighted_value:.4f}"
            )

        information_gain = parent_entropy - weighted_entropy

        print("\n" + "=" * 90)
        print(f"TOTAL WEIGHTED ENTROPY = {weighted_entropy:.4f}")
        print(
            f"INFORMATION GAIN = {parent_entropy:.4f} - "
            f"{weighted_entropy:.4f} = {information_gain:.4f}"
        )
        print("=" * 90)

        results.append({
            "Feature": feature,
            "Information Gain": information_gain
        })

    results_df = pd.DataFrame(results).sort_values(
        by="Information Gain", ascending=False
    ).reset_index(drop=True)

    print("\n\n" + "=" * 100)
    print("INFORMATION GAIN COMPARISON")
    print("=" * 100)
    print(results_df.to_string(index=False))

    best_feature = results_df.iloc[0]["Feature"]
    best_gain = results_df.iloc[0]["Information Gain"]

    print("\n" + "=" * 100)
    print("BEST SPLIT")
    print("=" * 100)
    print(f"Highest Information Gain Feature: {best_feature}")
    print(f"Information Gain: {best_gain:.4f}")

    return results_df


# PART 1 — Find the Root Node

At the beginning, all three feature columns are possible questions.

In [ ]:
features = [
    "Has High Graphics?",
    "Is it Multi-Player?",
    "Has Strong Marketing?"
]

root_results = compare_splits(df, features)


The feature with the highest Information Gain becomes the **root node**.

For this dataset, that should be **Has High Graphics?**

## 7. Create the two High Graphics branches

In [ ]:
graphics_yes = df[df["Has High Graphics?"] == "Yes"].copy()
graphics_no = df[df["Has High Graphics?"] == "No"].copy()

print("HIGH GRAPHICS = YES")
print("Record IDs:", graphics_yes["Record ID"].tolist())
print(graphics_yes["Hit/Flop"].value_counts())
print(f"Entropy = {entropy(graphics_yes):.4f}")

print("\nHIGH GRAPHICS = NO")
print("Record IDs:", graphics_no["Record ID"].tolist())
print(graphics_no["Hit/Flop"].value_counts())
print(f"Entropy = {entropy(graphics_no):.4f}")


`High Graphics = No` contains only Flops, so that branch is finished and becomes a **Flop leaf**.

The `High Graphics = Yes` branch is still mixed, so we continue splitting those records.

# PART 2 — Find the Next Node for High Graphics = Yes

In [ ]:
remaining_features = [
    "Is it Multi-Player?",
    "Has Strong Marketing?"
]

graphics_yes_results = compare_splits(
    graphics_yes,
    remaining_features
)


The highest Information Gain on this subset should be **Has Strong Marketing?**

## 8. Create the Strong Marketing branches

In [ ]:
marketing_yes = graphics_yes[graphics_yes["Has Strong Marketing?"] == "Yes"].copy()
marketing_no = graphics_yes[graphics_yes["Has Strong Marketing?"] == "No"].copy()

print("GRAPHICS = YES, MARKETING = YES")
print("Record IDs:", marketing_yes["Record ID"].tolist())
print(marketing_yes["Hit/Flop"].value_counts())
print(f"Entropy = {entropy(marketing_yes):.4f}")

print("\nGRAPHICS = YES, MARKETING = NO")
print("Record IDs:", marketing_no["Record ID"].tolist())
print(marketing_no["Hit/Flop"].value_counts())
print(f"Entropy = {entropy(marketing_no):.4f}")


`Marketing = No` is pure Hit, so that branch becomes a **Hit leaf**.

`Marketing = Yes` is mixed, so continue with the last unused feature.

# PART 3 — Final Split

In [ ]:
final_results = compare_splits(
    marketing_yes,
    ["Is it Multi-Player?"]
)


## 9. Create the final Multiplayer branches

In [ ]:
multiplayer_yes = marketing_yes[marketing_yes["Is it Multi-Player?"] == "Yes"].copy()
multiplayer_no = marketing_yes[marketing_yes["Is it Multi-Player?"] == "No"].copy()

print("MULTIPLAYER = YES")
print("Record IDs:", multiplayer_yes["Record ID"].tolist())
print(multiplayer_yes["Hit/Flop"].value_counts())
print(f"Entropy = {entropy(multiplayer_yes):.4f}")

print("\nMULTIPLAYER = NO")
print("Record IDs:", multiplayer_no["Record ID"].tolist())
print(multiplayer_no["Hit/Flop"].value_counts())
print(f"Entropy = {entropy(multiplayer_no):.4f}")


The `Multiplayer = Yes` branch is pure Flop.

The `Multiplayer = No` branch has 2 Hits and 1 Flop. There are no unused features left, so the leaf prediction is the majority class: **Hit**.

# 10. Print the final tree with Record IDs

In [ ]:
print("FINAL DECISION TREE")
print("===================")
print("Has High Graphics?")
print("├── No  [2, 4, 7, 9, 12, 14, 16, 18, 20] -> FLOP")
print("└── Yes [1, 3, 5, 6, 8, 10, 11, 13, 15, 17, 19]")
print("    └── Has Strong Marketing?")
print("        ├── No  [1, 5, 8, 10, 13, 15, 17] -> HIT")
print("        └── Yes [3, 6, 11, 19]")
print("            └── Is it Multi-Player?")
print("                ├── Yes [11] -> FLOP")
print("                └── No  [3, 6, 19] -> HIT (majority class)")


# 11. Draw the Decision Tree

Now use `scikit-learn` to draw a visual version of the tree.

In [ ]:
tree_data = df.copy()

tree_data["Has High Graphics?"] = tree_data["Has High Graphics?"].map({"Yes": 1, "No": 0})
tree_data["Is it Multi-Player?"] = tree_data["Is it Multi-Player?"].map({"Yes": 1, "No": 0})
tree_data["Has Strong Marketing?"] = tree_data["Has Strong Marketing?"].map({"Yes": 1, "No": 0})

X = tree_data[[
    "Has High Graphics?",
    "Is it Multi-Player?",
    "Has Strong Marketing?"
]]

y = tree_data["Hit/Flop"]

model = DecisionTreeClassifier(
    criterion="entropy",
    random_state=0
)

model.fit(X, y)


In [ ]:
plt.figure(figsize=(16, 9))

plot_tree(
    model,
    feature_names=X.columns,
    class_names=model.classes_,
    filled=True,
    rounded=True,
    fontsize=11
)

plt.title("Decision Tree - Game Success Prediction")
plt.show()


# 12. Predict every Record ID

This shows the actual class from the CSV and the class predicted by the completed tree.

In [ ]:
df_predictions = df[["Record ID", "Hit/Flop"]].copy()
df_predictions["Predicted"] = model.predict(X)
df_predictions["Correct?"] = df_predictions["Hit/Flop"] == df_predictions["Predicted"]

df_predictions


# 13. Show the exact path taken by one Record ID

Entropy chooses which feature becomes a node when the tree is built. After that, a record moves left or right according to its own feature value.

Use the function below to trace any Record ID.

In [ ]:
def trace_record(record_id):
    row_original = df[df["Record ID"] == record_id].iloc[0]
    row_encoded = X.loc[df["Record ID"] == record_id].iloc[0]

    print(f"TRACE FOR RECORD ID {record_id}")
    print("=" * 60)
    print(f"Actual class: {row_original['Hit/Flop']}")

    tree = model.tree_
    node = 0

    while tree.feature[node] != -2:
        feature_index = tree.feature[node]
        feature_name = X.columns[feature_index]
        threshold = tree.threshold[node]
        encoded_value = row_encoded[feature_name]
        original_value = row_original[feature_name]

        print(f"\nQuestion: {feature_name}")
        print(f"Record value: {original_value}")

        if encoded_value <= threshold:
            print("Direction: No / Left branch")
            node = tree.children_left[node]
        else:
            print("Direction: Yes / Right branch")
            node = tree.children_right[node]

    predicted_class = model.classes_[np.argmax(tree.value[node][0])]
    print(f"\nFINAL PREDICTION: {predicted_class}")


Try Record 11:

In [ ]:
trace_record(11)


You can replace `11` with any Record ID from 1 through 20, for example:

```python
trace_record(5)
trace_record(19)
```